In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

In [3]:
# ==========================================
# Your existing data
# X should have shape (n_samples, 3)
# y should have shape (n_samples,)
# ==========================================

# Example
X = np.load('function_3/initial_inputs.npy')
y = np.load('function_3/initial_outputs.npy')


X = np.vstack([X, [0.951374, 0.650589, 0.858494], 
               [0.929806, 0.916897, 0.397283],
               [0.054221, 0.235704, 0.644802],
               [0.958039, 0.932016, 0.513224],
               [0.059921, 0.235064, 0.066089],
               [0.047317, 0.871364, 0.365202],
               [0.046809, 0.219951, 0.861128],
               [0.965995, 0.221907, 0.341629],
               [0.047122, 0.245081, 0.502033],
               [0.047012, 0.280380, 0.502678],
               [0.047371, 0.278027, 0.501418]])

y = np.append(y, [-0.04461264702330381,
                  -0.04785883694771228,
                  -0.13026396473231236,
                  -0.04771819574901655,
                  -0.06833958436211048,
                  -0.06136989256250435,
                  -0.06344046194917227,
                  -0.06411672171104583,
                  -0.03662268216396786,
                  -0.025060602977315306,
                  -0.04388061263199139])

In [4]:
# ==========================================
# Fit Gaussian Process
# ==========================================

kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=np.ones(X.shape[1]), nu=2.5)
    + WhiteKernel(noise_level=1e-6)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0
)

gp.fit(X, y)

# ==========================================
# Expected Improvement
# ==========================================

def expected_improvement(x, gp, y_best, xi=0.001):
    x = np.array(x).reshape(1, -1)

    mu, sigma = gp.predict(x, return_std=True)

    sigma = sigma[0]
    mu = mu[0]

    if sigma < 1e-12:
        return 0.0

    improvement = mu - y_best - xi
    Z = improvement / sigma

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    return ei



# ==========================================
# Optimise acquisition function
# ==========================================

bounds = list(zip(X.min(axis=0), X.max(axis=0))) # replace with your limits

y_best = np.max(y)


def objective(x):
    return -expected_improvement(x, gp, y_best)


best_x = None
best_ei = -np.inf

for _ in range(10000):

    x0 = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds]
    )

    res = minimize(
        objective,
        x0=x0,
        bounds=bounds,
        method="L-BFGS-B"
    )

    ei = -res.fun

    if ei > best_ei:
        best_ei = ei
        best_x = res.x

print("Next point to evaluate:")
next_point = np.round(best_x, decimals=6)
print(next_point)

print("Expected Improvement:")
print(best_ei)

C:\Users\mho2\AppData\Roaming\Python\Python310\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Next point to evaluate:
[0.053934 0.263796 0.506931]
Expected Improvement:
0.0010725051119753752
